This notebook requires Infernal `.tbl` files produced by `cmsearch`.

In [ ]:
REAL_TBL = ""
GENERATED_TBL = ""

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica"],
    "axes.linewidth": 1.2,
    "axes.edgecolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
    "xtick.major.width": 1.2,
    "ytick.major.width": 1.2,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.labelsize": 12,
})

In [ ]:
TARGET_FAMILIES = {
    "RF00001": "5S",
    "RF00002": "5.8S",
    "RF01960": "18S",
    "RF02543": "28S",
}

RNA_ORDER = ["5S", "5.8S", "18S", "28S"]
GROUP_ORDER = ["Real", "Generated"]
PALETTE = {"Real": "#3C5488", "Generated": "#E64B35"}
YLIM_BOTTOMS = {
    "5S": 60,
    "5.8S": 100,
    "18S": 1000,
    "28S": 500,
}
E_VALUE_THRESHOLD = 1e-3

In [ ]:
def parse_and_filter_infernal_tbl(filepath: Path, group_name: str) -> pd.DataFrame:
    filepath = Path(filepath)
    records = []

    with filepath.open("r", encoding="utf-8") as handle:
        for line in handle:
            if line.startswith("#"):
                continue

            parts = line.split()
            if len(parts) < 16:
                continue

            rfam_acc = parts[3]
            if rfam_acc not in TARGET_FAMILIES:
                continue

            e_value = float(parts[15])
            if e_value >= E_VALUE_THRESHOLD:
                continue

            seq_from = int(parts[7])
            seq_to = int(parts[8])

            records.append({
                "Target": parts[0],
                "rRNA_Type": TARGET_FAMILIES[rfam_acc],
                "Length": abs(seq_to - seq_from) + 1,
                "Bit_Score": float(parts[14]),
                "E_value": e_value,
                "Group": group_name,
            })

    df = pd.DataFrame(records)

    if df.empty:
        return df

    return (
        df.sort_values("Bit_Score", ascending=False)
        .drop_duplicates(subset=["Target", "rRNA_Type"])
        .reset_index(drop=True)
    )

In [ ]:
df_real = parse_and_filter_infernal_tbl(REAL_TBL, "Real")
df_gen = parse_and_filter_infernal_tbl(GENERATED_TBL, "Generated")
df_all = pd.concat([df_real, df_gen], ignore_index=True)

print(f"Extraction complete. Real count: {len(df_real)} | Generated count: {len(df_gen)}")
if not df_all.empty:
    print("\nMerged distribution by type:\n", df_all.groupby(["rRNA_Type", "Group"]).size())

In [ ]:
def plot_violin_panels(df: pd.DataFrame) -> None:
    plot_order = [rna for rna in RNA_ORDER if rna in df["rRNA_Type"].unique()]

    fig, axes = plt.subplots(
        1,
        len(plot_order),
        figsize=(2.2 * len(plot_order), 4.5),
        dpi=300,
    )

    if len(plot_order) == 1:
        axes = [axes]

    for i, rna_type in enumerate(plot_order):
        ax = axes[i]
        subset = df[df["rRNA_Type"] == rna_type]

        sns.violinplot(
            data=subset,
            x="Group",
            y="Bit_Score",
            order=GROUP_ORDER,
            palette=PALETTE,
            inner="quartile",
            linewidth=1,
            width=0.7,
            cut=0,
            ax=ax,
        )

        if rna_type in YLIM_BOTTOMS:
            ax.set_ylim(bottom=YLIM_BOTTOMS[rna_type])

        ax.set_title(rna_type, fontsize=15, pad=12)
        ax.set_xlabel("")
        ax.set_xticklabels(GROUP_ORDER, fontsize=12)
        ax.tick_params(axis="y", labelsize=12)

        if i == 0:
            ax.set_ylabel("Infernal Bit Score", fontsize=15, labelpad=10)
        else:
            ax.set_ylabel("")

        sns.despine(ax=ax)
        ax.grid(False)

    plt.subplots_adjust(wspace=0.35)
    plt.show()

In [ ]:
if df_all.empty:
    print("No data were extracted. Please check the .tbl paths and parsing logic.")
else:
    plot_violin_panels(df_all)